# Denoise with NGMeet

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from EDX import *
from utils import *
from utils_sofima import *
import torch
from torch.utils.data import DataLoader
from model import UNet
from utils_noise import *
from bm3d import bm3d, BM3DStages
from bm4d import bm4d, BM4DStages
import pickle
import copy
import time
import humanfriendly
from skimage.restoration import estimate_sigma
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import random
import humanfriendly
from torchmetrics.image import SpectralAngleMapper
import pacmap
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns


device = torch.device("mps")
print(device)

%load_ext autoreload
%autoreload 2

### Load aligned tile (to be denoised)

In [ ]:
with open('../preprocessing_basic/results/preprocessed_edx/20251201_142554_tile_aligned_20frames.pkl', 'rb') as file:
    tile = pickle.load(file)

tile.apply("crop", parameters = {"crop_idx": (slice(50,tile.EDX_dim[0]-50),slice(50,tile.EDX_dim[1]-50),slice(None,None,None))})
tile.apply("binning", parameters={"dim": (974,974,250)})
tile.apply("MeanFilterEDX", parameters={"kernel_size": 3})

#h, w, b = tile.EDX_dim
tile.summary()

### (optional) Load non-aligned tile

In [ ]:
# load data
file_path = "../data/EMD/EDXdataset.emd"
EDX, haadf, xray_energies = load_EDX(file_path, first_frame=0, last_frame=100,sum_frames=True)
tile = EM_EDX(haadf, EDX, xray_energies)
tile.apply("crop", parameters={"crop_idx": (slice(None), slice(None), slice(96, 4096))})
#tile.apply("binning", parameters={"dim": (1024, 1024, 500)})
tile.apply("MeanFilterEDX", parameters={"kernel_size": 3})


### Load refrence tile

In [ ]:
with open('../preprocessing_basic/results/preprocessed_edx/20260114_062341_tile_aligned_100frames_align2zero.pkl', 'rb') as file:
    tile_ref = pickle.load(file)

tile_ref.apply("crop", parameters = {"crop_idx": (slice(50,tile_ref.EDX_dim[0]-50),slice(50,tile_ref.EDX_dim[1]-50),slice(None,None,None))})
tile_ref.apply("binning", parameters={"dim": (int(tile_ref.EDX_dim[0]/2),int(tile_ref.EDX_dim[1]/2),250)})
tile_ref.apply("MeanFilterEDX", parameters={"kernel_size": 3})

#h, w, b = tile.EDX_dim
tile_ref.summary()

### Optional: additional cropping

In [ ]:
tile.apply("crop", parameters = {"crop_idx": (slice(600,900),slice(600,900),slice(None,None,None))})
tile_ref.apply("crop", parameters = {"crop_idx": (slice(600,900),slice(600,900),slice(None,None,None))})

### NGMeet in matlab

In [ ]:
# Start matlab engine and set path
eng = matlab.engine.start_matlab()
matlab_path = eng.genpath('../matlab/')   # add path recursively
eng.addpath(matlab_path, nargout=0)

### (Optional) Get a 'Hyperspectral tile' from Matlab's libraries for reference

In [ ]:
datacube = eng.imhypercube("paviaU.hdr")
datacube = eng.gather(datacube)

# optional: denoise
datacube = eng.rescale(datacube)
datacube = eng.imnoise(datacube,"Gaussian") #,0,0.005)

datacube = np.asarray(datacube)
fake_haadf = np.sum(datacube,axis=2)
fake_xray_energies = np.ones((datacube.shape[2],1))
tile_pavia = EM_EDX(fake_haadf, datacube, fake_xray_energies)

In [ ]:
# Match their dimensions
tile.apply("crop", parameters = {"crop_idx": (slice(0,tile_pavia.EDX_dim[0]),slice(0,tile_pavia.EDX_dim[1]),slice(None,None,None))})

### Display

In [ ]:
# compare
bands_1 = [4,25,28]
bands_2 = [80,34,9]
f, ax = plt.subplots(1,2,figsize=(10,10))
ax[0].imshow(tile.FalseColor(bands_1))
ax[1].imshow(tile_pavia.FalseColor(bands_2))


### NGMeet parameters

In [ ]:
# set dataset to denoise
hsi_noisy = np.ascontiguousarray(tile.EDX)
hsi_noisy, min, max = MinMax(hsi_noisy, return_extra=True)
h,w,b = hsi_noisy.shape


# Optional variance stablilzation
G = np.mean(hsi_noisy,axis=2).reshape(h*w,1)
#G = np.ones_like(G)   # optional
H = np.mean(np.mean(hsi_noisy,axis=0),axis=0).reshape(b,-1)
W = G@np.transpose(H)
W = np.sqrt(W)    
hsi_noisy = np.divide(hsi_noisy,W.reshape((h,w,b)))


# Estimate noise
noise_est, noise_corr = est_noise(hsi_noisy.reshape(tile.EDX_2D.shape), 'additive_noise')

In [ ]:
noise_est.shape

### HySime to estimate k

In [ ]:
k, _ = hysime(hsi_noisy.reshape(tile.EDX_2D.shape), noise_est, noise_corr)
print(k)
sigma  = np.sqrt(np.var(noise_est))

In [ ]:
sigma_

### Default value for sigma in the 
- 0.1 times the standard deviation of a channel in the middle.

In [ ]:
middel_channel = int(np.ceil(tile.EDX_dim[2]/2))
print("Middle channel: ", middel_channel)
print("Before VTS: ", 0.1*np.std(tile.EDX[:,:,middel_channel]))
print("After VST: ", 0.1*np.std(hsi_noisy[:,:,middel_channel]))
print("Estimate (psp): ", sigma)
print("Skimage estimate: ", estimate_sigma(hsi_noisy))

### Denoising

In [ ]:
start = time.perf_counter()



hsi_clean =  eng.denoiseNGMeet(
    hsi_noisy,     
    'Sigma', 0.3,   #sigma,
    'SpectralSubspace', 10,
    'NumIterations', 2,
    nargout=1
)
hsi_clean =  np.asarray(hsi_clean)


# optional undo variance scaling (and minmaxing)
hsi_clean = np.multiply(hsi_clean, W.reshape((h,w,b)))
hsi_clean = MinMaxInverse(hsi_clean, min, max)




# Assign to new tile object
tile_NGMeet = tile.apply("crop", parameters={"crop_idx": (slice(None), slice(None), slice(None))},copy_instance=True)
tile_NGMeet.EDX = hsi_clean


# time
elapsed = time.perf_counter() - start
print("Elapsed time for denoising:", humanfriendly.format_timespan(elapsed))

In [ ]:
# compare
corr = 0


#bands = [25-corr,25-corr,25-corr]
#bands = [28-corr,28-corr,28-corr]
#bands = [4-corr,4-corr,4-corr]


bands = [4-corr,25-corr,28-corr]


#zoom = (slice(400,600),slice(150,350),slice(None))
zoom = (slice(600,900),slice(600,900),slice(None))
#zoom = (slice(None,None),slice(None,None),slice(None))
#zoom = (slice(1250,1750),slice(1250,1750),slice(None))

f, ax = plt.subplots(2,3,figsize=(15,10))
ax[0][0].imshow(tile.FalseColor(bands))
ax[0][0].set_title('Raw')
ax[0][1].imshow(tile_NGMeet.FalseColor(bands))
ax[0][1].set_title('Denoised')
ax[0][2].imshow(tile_ref.FalseColor(bands))
ax[0][2].set_title('Ref (100 aligned)')

ax[1][0].imshow(tile.FalseColor(bands)[zoom])
ax[1][1].imshow(tile_NGMeet.FalseColor(bands)[zoom])
ax[1][2].imshow(tile_ref.FalseColor(bands)[zoom])

### PSNR comparison

In [ ]:
f, ax = plt.subplots(figsize=(15,15))

ax.plot(eval_psnr(MinMax(tile.EDX).astype('float32'),MinMax(tile_ref.EDX).astype('float32')))
ax.plot(eval_psnr(MinMax(tile_NGMeet.EDX).astype('float32'),MinMax(tile_ref.EDX).astype('float32')),'r')

### Spectral Angle Mapper

In [ ]:
SAM_torch = SpectralAngleMapper(reduction='none')
t1 = torch.from_numpy(MinMax(tile.EDX).transpose(2,0,1)).unsqueeze(0)
t2 = torch.from_numpy(MinMax(tile_ref.EDX).transpose(2,0,1)).unsqueeze(0)

metric_noisy = SAM_torch(t1,t2).squeeze().numpy()

t1 = torch.from_numpy(MinMax(tile_NGMeet.EDX).transpose(2,0,1)).unsqueeze(0)
t2 = torch.from_numpy(MinMax(tile_ref.EDX).transpose(2,0,1)).unsqueeze(0)
metric_denoised = SAM_torch(t1,t2).squeeze().numpy()




In [ ]:
zoom = (slice(600,None),slice(None,400))

f, ax = plt.subplots(2,2,figsize=(15,15))
ax[0][0].imshow(metric_noisy,cmap='jet')
ax[0][0].set_title(f"Noisy average SAM with reference {metric_noisy.mean():.4f}")
ax[0][1].imshow(metric_denoised,cmap='jet')
ax[0][1].set_title(f"NGMeet average SAM with reference {metric_denoised.mean():.4f}")
ax[1][0].imshow(metric_noisy[zoom],cmap='jet')
ax[1][1].imshow(metric_denoised[zoom],cmap='jet')




### PACMAP

In [ ]:
tiles_binned = [tile,tile_NGMeet,tile_ref]

n_samples = tiles_binned[0].EDX.shape[0]**2

# subsample
subsample_ratio = 0.3
nTrain = int(subsample_ratio*n_samples)   # number of training samples
multi_scale = False
poisson = False

# random seed
random.seed(100)
sample_idx = np.random.choice(n_samples,nTrain,replace=False)  

all_embeddings = []

for tile in tiles_binned:

    start = time.perf_counter()
    
    # PaCMAP
    reduction_model = pacmap.PaCMAP(n_components=2, n_neighbors=None, save_tree=True)

    # select the training dataset (w/o any scaling/normalizations)
    hsi_train = tile.EDX

    # get dimensions
    h, w, b = tile.EDX_dim
    
    # options (in this order)
    if poisson:       
        # Optional variance stablilzation
        G = np.mean(tile.EDX,axis=2).reshape(h*w,1)
        H = np.mean(np.mean(tile.EDX,axis=0),axis=0).reshape(b,-1)
        W = G@np.transpose(H)
        W = np.sqrt(W)    
        hsi_train = np.divide(tile.EDX,W.reshape((h,w,b)))
    
    if multi_scale:
        hsi_train = hsi_multi_scale(hsi_train,radii=[1,3,5],sigma=2)

    # fitting
    hsi_train = hsi_train.reshape((h*w,hsi_train.shape[2]))  
    reduction_model.fit(hsi_train[sample_idx,:])
    embeddings = reduction_model.embedding_
    
    # scaling (scale both reduced dimensions to 0-1)
    scaler = MinMaxScaler()
    embeddings = scaler.fit_transform(embeddings)

    # append to list
    all_embeddings.append(embeddings)

    # time
    elapsed = time.perf_counter() - start
    print(f"Elapsed time for finding pacmap embeddings: {humanfriendly.format_timespan(elapsed):s}")

In [ ]:
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

# Get masks
# get the hand-annotated masks and display them
mask_dir = os.path.join(os.path.dirname(os.getcwd()),'data', 'masks_frame0_sil_specific')
masks = create_masks(mask_dir)
colors = ['k','r','g','b','c','m','y','w']
newcmap = ListedColormap(colors)
plt.imshow(masks,cmap=newcmap)
plt.show()

# crop 
masks = masks[25:1024-25,25:1024-25]

In [ ]:
%matplotlib inline 

titles = ['20 frames aligned','20 frames aligned + NGMeet', '100 frames aligned']

f, ax = plt.subplots(1,3,figsize=(30,10))
sns.despine(left=True, bottom=True, right=True)


# colors
colors = ['k', 'r', 'g', 'b', 'c', 'm', 'y']
plt_colors = [colors[i] for i in masks.reshape((974**2))[sample_idx]]

# marker size
marker_size = []
for i in range(len(plt_colors)):
    if plt_colors[i] == 'k':  
        marker_size.append(0.0001)
    else:
        marker_size.append(1)

plt_idx = 0
for i in range(len(tiles_binned)):
    ax[i].set_facecolor((1,1,1)) 
    ax[i].scatter(all_embeddings[plt_idx][:,0],all_embeddings[plt_idx][:,1], s=marker_size,c=plt_colors) 
    ax[i].axis('off')
    ax[i].set_title(titles[plt_idx],fontsize=18)
    plt_idx+=1


f.tight_layout()
plt.show()

### Silhouette comparison

In [ ]:
# compute sil scores for both
metric = 'euclidean'

sil_img0 = sil_scores(tiles_binned[0].EDX, masks, metric=metric)
sil_img1 = sil_scores(tiles_binned[1].EDX, masks, metric=metric)
sil_img2 = sil_scores(tiles_binned[2].EDX, masks, metric=metric)



# compute global vmin/vmax 
vmin = np.nanmin(np.array([sil_img0,sil_img1,sil_img2]))
vmax = np.nanmax(np.array([sil_img0,sil_img1,sil_img2]))

In [ ]:
cmap = plt.cm.jet.copy()
cmap.set_bad(color='black')

bands=[24,24,24]
zoom = (slice(250,450),slice(150,350),slice(None))

f, ax = plt.subplots(2,3,figsize=(15, 15))


ax[0][0].imshow(tiles_binned[0].FalseColor(bands)[zoom])
ax[0][1].imshow(tiles_binned[1].FalseColor(bands)[zoom])
ax[0][2].imshow(tiles_binned[2].FalseColor(bands)[zoom])
#ax[0][3].imshow(tiles_binned[3].FalseColor(bands))

ax[0][0].set_title(titles[0],fontsize=20)
ax[0][1].set_title(titles[1],fontsize=20)
ax[0][2].set_title(titles[2],fontsize=20)
#ax[0][3].set_title(titles[3],fontsize=20)

im = ax[1][0].imshow(sil_img0,cmap=cmap,vmin=vmin,vmax=vmax)
ax[1][0].set_title("Avg. sillhouette %.2f" % np.nanmean(sil_img0),fontsize=12)

im = ax[1][1].imshow(sil_img1,cmap=cmap,vmin=vmin,vmax=vmax)
ax[1][1].set_title("Avg. sillhouette %.2f" % np.nanmean(sil_img1),fontsize=12)

im = ax[1][2].imshow(sil_img2,cmap=cmap,vmin=vmin,vmax=vmax)
ax[1][2].set_title("Avg. sillhouette %.2f" % np.nanmean(sil_img2),fontsize=12)

#im = ax[1][3].imshow(sil_img3,cmap=cmap,vmin=vmin,vmax=vmax)
#ax[1][3].set_title("Avg. sillhouette %.2f" % np.nanmean(sil_img3),fontsize=12)

for i in range(2):
    for j in range(3):
        ax[i][j].axis('off')

# one shared colorbar

plt.tight_layout(pad=0.3)
plt.colorbar(im, ax=ax.ravel().tolist(), shrink=0.2,location='top')